# pepsy Quickstart

This notebook shows a clean end-to-end workflow:
1. Build a small PEPS ket.
2. Prepare `(ket, norm)` with `prepare_boundary_inputs`.
3. Build boundary MPS objects with `BdyMPS`.
4. Contract with `ContractBoundary` and inspect fidelity history.

In [ ]:
import numpy as np
import quimb.tensor as qtn
import pepsy

print('pepsy version:', pepsy.__version__)

In [ ]:
# Build a small random PEPS.
# Keep sizes modest so the notebook runs quickly on CPU.
ket = qtn.PEPS.rand(Lx=3, Ly=3, bond_dim=2, seed=1, dtype='complex128')
ket

In [ ]:
# Prepare the tagged ket and double-layer norm network.
# If bra is omitted, it is auto-built as ket.conj() with safe inner-index remapping.
ket_tagged, norm = pepsy.prepare_boundary_inputs(ket=ket)

print('ket tags sample:', sorted(list(ket_tagged.tags))[:6])
print('norm tensor count:', norm.num_tensors)

In [ ]:
# Build boundary states used by the contraction sweeps.
# chi controls boundary MPS bond dimension (accuracy vs cost).
bdy = pepsy.BdyMPS(
    tn_flat=ket_tagged,
    tn_double=norm,
    chi=32,
    single_layer=False,
)

print('number of boundary keys:', len(bdy.mps_b))
print('available keys:', bdy.available_boundary_keys())

In [ ]:
# Run boundary contraction and request structured results.
# fidel_=True records per-step fidelity values in result.fidel.
res = pepsy.ContractBoundary(
    norm=norm,
    mps_boundaries=bdy.mps_b,
    direction='y',
    n_iter=2,
    max_separation=0,
    fidel_=True,
    pbar=True,
)

print('cost:', res.cost)
print('fidelity list:', res.fidel)

In [ ]:
# Optional: inspect left/right fidelity products for the chosen direction.
# For direction='y', split point is y_left = Ly // 2 when max_separation=0.
split = ket.Ly // 2
f_left = np.prod(res.fidel[:split]) if split > 0 else 1.0
f_right = np.prod(res.fidel[split:]) if split < len(res.fidel) else 1.0

print('left product:', f_left)
print('right product:', f_right)